In [3]:
# ==============================
# 1. Import Libraries
# ==============================

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


# ==============================
# 2. Load Dataset
# ==============================

# The 'data' DataFrame is already available from previous cells.
# No need to load 'your_dataset.csv'.
data = pd.read_csv("/content/battery_cycle_level_dataset_CLEAN_FINAL.csv")

# Example expected columns
# battery_id, cycle, voltage, current, temperature, capacity

# print(data.head())


# ==============================
# 3. Calculate SOH
# ==============================

# Initial capacity per battery
initial_capacity = data.groupby("battery_id")["capacity"].transform("first")

data["SOH"] = data["capacity"] / initial_capacity


# ==============================
# 4. Calculate RUL
# ==============================

max_cycle = data.groupby("battery_id")["cycle"].transform("max")

data["RUL"] = max_cycle - data["cycle"]


# ==============================
# 5. Feature Selection
# ==============================

features = [
    "cycle",
    "voltage",
    # "current", # Current is not present in combined data
    "temperature"
]

X = data[features]

y_soh = data["SOH"]
y_rul = data["RUL"]


# ==============================
# 6. Feature Scaling
# ==============================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# ==============================
# 7. Models
# ==============================

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    )
}


# ==============================
# 8. Cross Validation
# ==============================

kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = {
    name: {
        "soh_rmse": [],
        "soh_r2": [],
        "rul_rmse": [],
        "rul_r2": []
    }
    for name in models
}


# ==============================
# 9. Training Loop
# ==============================

fold = 1

for train_index, test_index in kf.split(X_scaled):

    print(f"Fold {fold}")

    X_train, X_test = X_scaled[train_index], X_scaled[test_index]

    y_train_soh, y_test_soh = y_soh.iloc[train_index], y_soh.iloc[test_index]
    y_train_rul, y_test_rul = y_rul.iloc[train_index], y_rul.iloc[test_index]

    for name, model in models.items():

        # --- Train SOH model ---
        model.fit(X_train, y_train_soh)
        pred_soh = model.predict(X_test)

        # --- Train RUL model ---
        model.fit(X_train, y_train_rul)
        pred_rul = model.predict(X_test)

        # --- Metrics ---
        soh_rmse = np.sqrt(mean_squared_error(y_test_soh, pred_soh))
        soh_r2 = r2_score(y_test_soh, pred_soh)

        rul_rmse = np.sqrt(mean_squared_error(y_test_rul, pred_rul))
        rul_r2 = r2_score(y_test_rul, pred_rul)

        results[name]["soh_rmse"].append(soh_rmse)
        results[name]["soh_r2"].append(soh_r2)
        results[name]["rul_rmse"].append(rul_rmse)
        results[name]["rul_r2"].append(rul_r2)

    fold += 1


# ==============================
# 10. Average Results
# ==============================

print("\nAverage Results:\n")

for name in models:

    print(name)
    print(f"  SOH RMSE: {np.mean(results[name]['soh_rmse']):.4f}")
    print(f"  SOH R2: {np.mean(results[name]['soh_r2']):.4f}")
    print(f"  RUL RMSE: {np.mean(results[name]['rul_rmse']):.4f}")
    print(f"  RUL R2: {np.mean(results[name]['rul_r2']):.4f}\n")

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

Average Results:

Random Forest
  SOH RMSE: 0.0262
  SOH R2: 0.9511
  RUL RMSE: 7.0818
  RUL R2: 0.9710

XGBoost
  SOH RMSE: 0.0272
  SOH R2: 0.9466
  RUL RMSE: 8.4744
  RUL R2: 0.9560

